# Resynex DTG — ADNI cross-disease transfer, 5-seed ensemble

Trains the same architecture and the same training pipeline used for the
primary Parkinson's (PPMI) model on the Alzheimer's registry (ADNI) instead,
initialized fresh and trained on ADNI alone — not fine-tuned from the PPMI
checkpoint. Only `n_baseline`/`n_outcomes` differ, read directly from the
ADNI data. Six outcomes: CDRSB, MMSCORE, TOTAL13, ABETA, VENT\_NORM, HIPPO\_NORM.

This notebook trains **5 independent seeds** (42, 123, 456, 789, 1000) and
evaluates them as an ensemble (averaged point predictions/survival, mixture-
sampled generative draws), the same combination rule used for PPMI.

**Data you need to attach as a Kaggle Dataset before running this** (ADNI is
restricted-access and cannot be redistributed — request access at
https://adni.loni.usc.edu/ under your own Data Use Agreement, then upload the
eight `.rda` files below, all from the ADNIMERGE2 export, as a private Kaggle
Dataset and attach it to this notebook):

```
DXSUM.rda
CDR.rda
MMSE.rda
ADAS.rda
UPENNBIOMK_MASTER.rda
UCSFFSX.rda
PTDEMOG.rda
APOERES.rda
```

Set `ADNI_DATASET_SLUG` below to whatever you name that Kaggle Dataset.

## 1. Setup — clone both repositories and install dependencies

In [ ]:
ADNI_DATASET_SLUG = "adni-adnimerge2-raw"  # <-- change to your Kaggle Dataset's slug

# The training script imports PlatformDTG from the platform repo's own
# inference module, so a model trained here can never drift from the class
# the deployed service reconstructs to load it — clone it as a sibling.
!git clone --depth 1 https://github.com/SeifIslemBenrabah/Resynex-DTG-Model.git
!git clone --depth 1 https://github.com/SeifIslemBenrabah/Resynex-Platform.git
%cd Resynex-DTG-Model
!pip install -q -r requirements.txt

## 2. Point the training code at the data

`experiments/adni/adni_dataset_notau.py` reads `ADNI_DATA_DIR`
(defaults to `data/adni/ADNIMERGE2/data/` if unset) and
`model_platform_dtg.py` reads `RESYNEX_PLATFORM_PATH` to find the sibling
platform repo's `ms-digital-twin/` directory.

In [ ]:
import os
os.environ["ADNI_DATA_DIR"] = f"/kaggle/input/{ADNI_DATASET_SLUG}"
os.environ["RESYNEX_PLATFORM_PATH"] = "/kaggle/working/Resynex-Platform/ms-digital-twin"
assert os.path.isdir(os.environ["ADNI_DATA_DIR"]), (
    "Dataset not found — attach it via the notebook's Add Data panel "
    "and check ADNI_DATASET_SLUG matches its slug."
)
os.listdir(os.environ["ADNI_DATA_DIR"])

## 3. Load the raw cohort and visualize it

Same senior-ML-engineer check as for PPMI: look at the diagnosis mix, the
per-outcome missingness, and the marginal distribution of every outcome
before spending a training run on it.

In [ ]:
import sys; sys.path.insert(0, "experiments/adni")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from adni_dataset_notau import build, BASELINE_COLS, OUTCOME_COLS

X, Y, Tf, PID, ET, EI = build()
print(f"visit-pairs: {X.shape[0]}   baseline features: {X.shape[1]}   outcomes: {Y.shape[1]}")
print(f"unique patients: {len(np.unique(PID))}")

In [ ]:
# Missingness per outcome — this is what the imputer and the observation
# mask in the loss have to handle
obs_rate = (~np.isnan(Y)).mean(axis=0)
plt.figure(figsize=(6, 3.5))
plt.bar(OUTCOME_COLS, obs_rate, color="#55a868")
plt.ylabel("fraction observed"); plt.ylim(0, 1)
plt.title("Observation rate per outcome (visit-pairs)")
plt.xticks(rotation=30, ha="right"); plt.grid(alpha=.3, axis="y"); plt.show()

In [ ]:
# Marginal distribution of each of the 6 outcomes
fig, axes = plt.subplots(2, 3, figsize=(12, 6)); axes = axes.ravel()
for ax, col in zip(axes, OUTCOME_COLS):
    j = OUTCOME_COLS.index(col)
    vals = Y[:, j][~np.isnan(Y[:, j])]
    ax.hist(vals, bins=30, color="#c44e52")
    ax.set_title(f"{col}  (n={len(vals)})")
plt.suptitle("Distribution of the 6 ADNI prediction targets")
plt.tight_layout(); plt.show()

In [ ]:
# Diagnosis mix at the visits used (CN / MCI / Dementia), read off the
# curr_DX_NUM input-only column added by BASELINE_COLS
dx_idx = BASELINE_COLS.index("curr_DX_NUM")
dx_vals = X[:, dx_idx]
labels = {0.0: "CN", 1.0: "MCI", 2.0: "Dementia"}
counts = pd.Series(dx_vals).map(labels).value_counts()
plt.figure(figsize=(4.5, 3.5))
plt.bar(counts.index, counts.values, color="#8172b2")
plt.ylabel("visit-pairs"); plt.title("Diagnosis mix"); plt.show()

In [ ]:
# Survival curve for the same discrete-time event head used on PPMI
from lifelines import KaplanMeierFitter
kmf = KaplanMeierFitter()
kmf.fit(ET, event_observed=EI)
ax = kmf.plot_survival_function(figsize=(6, 4))
ax.set_xlabel("normalised time to event"); ax.set_ylabel("event-free probability")
ax.set_title(f"{int(EI.sum())}/{len(EI)} visit-pairs observed to reach the event")
plt.grid(alpha=.3); plt.show()

## 4. Train the 5-seed ensemble

Same architecture and same training pipeline as the PPMI configuration,
initialized fresh and trained on ADNI alone; only `n_baseline`/`n_outcomes`
differ, read directly from the data. Model capacity ($n_h=32$) is kept
identical to the PPMI configuration on purpose — enlarging it for this
registry specifically would answer a different question than whether the
architecture transfers without redesign.

In [ ]:
import subprocess, sys

SEEDS = [42, 123, 456, 789, 1000]
CKPT_DIRS = [f"models_out/adni_notau_s{s}" for s in SEEDS]

for s, out_dir in zip(SEEDS, CKPT_DIRS):
    print(f"\n===== training seed={s} =====")
    cmd = [sys.executable, "experiments/adni/train_adni_notau.py",
           "--out", out_dir, "--epochs", "60", "--warmup", "10",
           "--lam_var", "0.3", "--lam_cd", "0.1", "--nh", "32",
           "--seed", str(s)]
    subprocess.run(cmd, check=True)

## 5. Evaluate the ensemble

`--split_seed 42` matches the project's own held-out test split convention
(`ensemble_adni_notau.py`), so this reproduces the exact split the reported
C-index/R² numbers were computed on.

In [ ]:
ckpts = ",".join(f"{d}/model.pt" for d in CKPT_DIRS)
cmd = [sys.executable, "experiments/adni/metrics_adni_calibration_ensemble.py",
       "--ckpts", ckpts, "--split_seed", "42", "--nh", "32",
       "--out", "metrics_adni_ensemble.json"]
subprocess.run(cmd, check=True)

In [ ]:
import json
with open("metrics_adni_ensemble.json") as f:
    results = json.load(f)
print(json.dumps(results, indent=2))

## 6. Save the ensemble checkpoints and metrics

In [ ]:
import shutil, os
os.makedirs("/kaggle/working/adni_ensemble", exist_ok=True)
for s, d in zip(SEEDS, CKPT_DIRS):
    shutil.copy(f"{d}/model.pt", f"/kaggle/working/adni_ensemble/seed{s}.pt")
shutil.copy("metrics_adni_ensemble.json", "/kaggle/working/adni_ensemble/metrics_adni_ensemble.json")
print(os.listdir("/kaggle/working/adni_ensemble"))